# Exercício 5 — Projeto Integrado de Limpeza

Pergunta de negócio: como comparar preço, desconto e avaliação dos produtos para identificar categorias com maior potencial de venda?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

pd.set_option("display.max_columns", None)
arquivos = glob.glob("*.csv")
arquivos = sorted(set(arquivos))
[(arquivo, pd.read_csv(arquivo, nrows=3).shape[1]) for arquivo in arquivos[:10]]

## Seleção e carregamento

In [ ]:
if not arquivos:
    raise FileNotFoundError("Coloque pelo menos dois CSVs do dataset nesta pasta")
selecionados = arquivos[:2]
frames = []
for arquivo in selecionados:
    base = pd.read_csv(arquivo, low_memory=False)
    base["arquivo_origem"] = Path(arquivo).name
    frames.append(base)
df_original = pd.concat(frames, ignore_index=True, sort=False)
df_original.head()

## Diagnóstico inicial

In [ ]:
def indicadores(df):
    return pd.Series({
        "linhas": len(df),
        "colunas": df.shape[1],
        "percentual_ausentes": df.isna().mean().mean() * 100,
        "duplicatas": int(df.duplicated().sum()),
        "tipos": df.dtypes.astype(str).value_counts().to_dict()
    })
indicadores(df_original)

## Funções do pipeline

In [ ]:
def localizar_coluna(df, termos):
    for coluna in df.columns:
        normalizada = str(coluna).lower().replace("_", " ")
        if any(termo in normalizada for termo in termos):
            return coluna
    return None

def corrigir_tipos(df):
    resultado = df.copy()
    for coluna in resultado.columns:
        if resultado[coluna].dtype == "object":
            serie = resultado[coluna].astype("string").str.replace(r"[^0-9,.-]", "", regex=True).str.replace(",", ".", regex=False)
            convertido = pd.to_numeric(serie, errors="coerce")
            if convertido.notna().mean() >= 0.7:
                resultado[coluna] = convertido
    return resultado

def tratar_ausentes(df):
    resultado = df.copy()
    for coluna in resultado.select_dtypes(include=np.number):
        resultado[coluna] = resultado[coluna].fillna(resultado[coluna].median())
    for coluna in resultado.select_dtypes(include=["object", "string"]):
        resultado[coluna] = resultado[coluna].fillna("não informado")
    return resultado

def tratar_duplicatas(df):
    return df.drop_duplicates().copy()

def padronizar_textos(df):
    resultado = df.copy()
    for coluna in resultado.select_dtypes(include=["object", "string"]):
        resultado[coluna] = resultado[coluna].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()
    return resultado

def tratar_outliers(df):
    resultado = df.copy()
    for coluna in resultado.select_dtypes(include=np.number):
        q1, q3 = resultado[coluna].quantile([.25, .75])
        iqr = q3 - q1
        if iqr > 0:
            resultado[coluna] = resultado[coluna].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return resultado

def criar_features(df):
    resultado = df.copy()
    coluna_preco = localizar_coluna(resultado, ["price", "preço", "amount"])
    coluna_desconto = localizar_coluna(resultado, ["discount", "desconto"])
    coluna_avaliacao = localizar_coluna(resultado, ["rating", "review", "avaliacao"])
    if coluna_preco and coluna_desconto:
        resultado["preco_com_desconto"] = resultado[coluna_preco] * (1 - resultado[coluna_desconto].clip(0, 100) / 100)
    if coluna_avaliacao:
        resultado["produto_bem_avaliado"] = resultado[coluna_avaliacao] >= resultado[coluna_avaliacao].median()
    return resultado

## Execução do pipeline

In [ ]:
df_corrigido_tipos = corrigir_tipos(df_original)
df_sem_ausentes = tratar_ausentes(df_corrigido_tipos)
df_sem_duplicatas = tratar_duplicatas(df_sem_ausentes)
df_padronizado = padronizar_textos(df_sem_duplicatas)
df_sem_outliers = tratar_outliers(df_padronizado)
df_limpo = criar_features(df_sem_outliers)
df_limpo.head()

## Tabela antes x depois

In [ ]:
antes_depois = pd.concat({"antes": indicadores(df_original), "depois": indicadores(df_limpo)}, axis=1)
antes_depois

## Visualizações

In [ ]:
numericas_antes = df_original.select_dtypes(include=np.number).columns
numericas_depois = df_limpo.select_dtypes(include=np.number).columns
coluna_visual = next((c for c in numericas_depois if c in numericas_antes), None)
if coluna_visual:
    fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_original[coluna_visual].dropna(), bins=40, ax=eixos[0])
    sns.histplot(df_limpo[coluna_visual].dropna(), bins=40, ax=eixos[1])
    eixos[0].set_title("Antes")
    eixos[1].set_title("Depois")
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(10, 4))
antes_depois.loc[["percentual_ausentes", "duplicatas"]].T.plot(kind="bar")
plt.tight_layout()
plt.show()

## Relatório e respostas de reflexão

O problema mais difícil de perceber costuma ser um número armazenado como texto, porque a coluna pode parecer preenchida e os valores podem ser exibidos normalmente. O erro só aparece quando se tenta calcular estatísticas ou comparar produtos.

A ordem importa. Se os outliers forem tratados antes de preencher ausentes, os limites podem ser calculados em uma amostra menor e produzir limites diferentes. Se a imputação usar valores extremos ainda não tratados, a mediana ou a média também pode ser afetada, dependendo da escolha.

Se a pergunta de negócio fosse detectar fraude, produtos com comportamento extremo ou erros de catálogo, eu mudaria a decisão de aplicar capping indiscriminadamente. Nesse caso, os extremos deveriam ser sinalizados para investigação, não substituídos automaticamente.